In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 스타벅스 데이터셋 3개 로드
portfolio = pd.read_csv("portfolio.csv")
profile = pd.read_csv("profile.csv")
transcript = pd.read_csv("transcript.csv")


##transcript의 value 컬럼 파싱
# transcript의 value 컬럼에는 offer_id, reward, amount 등이 딕셔너리 형태로 섞여있다.
# 이를 분리하여 분석 가능한 컬럼으로 만든다.

def extract_offer_id(x):
    if isinstance(x, dict):
        if 'offer id' in x:
            return x['offer id']
        elif 'offer_id' in x:
            return x['offer_id']
    return None

def extract_amount(x):
    if isinstance(x, dict) and 'amount' in x:
        return x['amount']
    return None

transcript['offer_id'] = transcript['value'].apply(extract_offer_id)
transcript['amount'] = transcript['value'].apply(extract_amount)



## 이벤트별 고객 행동 집계
# 고객별 이벤트 횟수를 계산한다.

event_counts = transcript.pivot_table(
    index='person',
    columns='event',
    aggfunc='size',
    fill_value=0
)

# 컬럼 이름 정리
event_counts = event_counts.rename(columns={
    'offer received': 'offer_received',
    'offer viewed': 'offer_viewed',
    'offer completed': 'offer_completed',
    'transaction': 'transaction_count'
})



# 고객 총 구매금액 계산
# transaction 이벤트의 amount를 합산하여 고객별 총 소비금액 계산
spending = transcript.groupby('person')['amount'].sum()

## 고객 행동 테이블 생성
customer_behavior = event_counts.join(spending)
customer_behavior = customer_behavior.rename(columns={
    'amount': 'total_spending'
})


## Feature Engineering
# 프로모션 반응률
customer_behavior['response_rate'] = (
    customer_behavior['offer_completed'] /
    customer_behavior['offer_received']
)

# 프로모션 조회율
customer_behavior['view_rate'] = (
    customer_behavior['offer_viewed'] /
    customer_behavior['offer_received']
)

# NaN 처리
customer_behavior['response_rate'] = customer_behavior['response_rate'].fillna(0)
customer_behavior['view_rate'] = customer_behavior['view_rate'].fillna(0)


##  profile 데이터 전처리
# membership 시작 날짜 변환
profile['became_member_on'] = pd.to_datetime(
    profile['became_member_on'], format='%Y%m%d'
)

# 멤버십 가입 후 경과 기간 계산
profile['membership_age'] = (
    pd.Timestamp("2018-01-01") - profile['became_member_on']
).dt.days


##  Loyalty Score 생성
# 고객 충성도를 간단히 측정하기 위한 지표
# 구매 횟수 + 소비금액 기반
customer_behavior['loyalty_score'] = (
    customer_behavior['transaction_count'] +
    customer_behavior['total_spending'] / 100
)


# 고객 데이터 결합
customer_data = customer_behavior.merge(
    profile,
    left_index=True,
    right_on='id'
)


# 고객 세그먼트 생성 ／행동 기반 고객 세분화

conditions = [
    customer_data['response_rate'] > 0.6,  # 프로모션 반응 높은 고객
    customer_data['transaction_count'] > 20,  # 구매 빈도 높은 고객
    customer_data['transaction_count'] < 5   # 활동 적은 고객
]

choices = [
    'Promotion-sensitive',
    'Loyal',
    'Passive'
]

customer_data['segment'] = np.select(conditions, choices, default='General')


# 세그먼트별 요약 분석
segment_summary = customer_data.groupby('segment').agg({
    'transaction_count': 'mean',
    'total_spending': 'mean',
    'response_rate': 'mean'
})

print(segment_summary)


# 고객 연령대를 구간화
customer_data['age_group'] = pd.cut(
    customer_data['age'],
    bins=[0,20,30,40,50,60,100],
    labels=['<20','20s','30s','40s','50s','60+']
)



# 연령대별 프로모션 선호 분석
# 고객이 가장 많이 완료한 offer type을 계산하기 위한 준비

completed = transcript[transcript['event'] == 'offer completed']

completed = completed.merge(
    portfolio,
    left_on='offer_id',
    right_on='id'
)

completed = completed.merge(
    profile,
    left_on='person',
    right_on='id'
)

# 연령대별 offer type 완료 횟수
age_offer = completed.groupby(
    ['age_group','offer_type']
).size().reset_index(name='count')



## 캠페인 반응 시간 분석
# offer received → viewed까지 걸리는 시간 분석

received = transcript[transcript['event'] == 'offer received']
viewed = transcript[transcript['event'] == 'offer viewed']

response_time = viewed['time'].values - received['time'].values[:len(viewed)]


## 시각화

# 고객 연령 분포
plt.figure(figsize=(8,5))
sns.histplot(customer_data['age'], bins=30)
plt.title("Customer Age Distribution")
plt.show()


# 세그먼트 분포
plt.figure(figsize=(6,4))
customer_data['segment'].value_counts().plot(kind='bar')
plt.title("Customer Segments")
plt.show()


# 세그먼트별 소비금액
plt.figure(figsize=(8,5))
sns.boxplot(
    x='segment',
    y='total_spending',
    data=customer_data
)
plt.title("Spending by Customer Segment")
plt.show()


# 연령대별 프로모션 선호
plt.figure(figsize=(8,5))
sns.barplot(
    x='age_group',
    y='count',
    hue='offer_type',
    data=age_offer
)
plt.title("Promotion Preference by Age Group")
plt.show()